In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="09-similar-listing",
    notebook_name="04_interview_walkthrough.ipynb"
)

# Similar Listings: Complete Interview Walkthrough

## What This Notebook Is

This is a **complete mock interview simulation** for the Similar Listings system design problem (Airbnb-style). It is written as a dialogue between an interviewer and a staff-level candidate, with timing guidance, scoring rubrics, and the exact key phrases that earn top scores.

Think of this as your **dress rehearsal**. Read the candidate's lines out loud. Practice until the structure feels natural, then adapt it to your own words.

---

## Interview Structure (45 Minutes)

| Phase | Time | What Happens |
|-------|------|-------------|
| 1. Clarify Requirements | 3-5 min | Ask smart questions, define scope |
| 2. Frame as ML Problem | 3-5 min | Session-based rec, embedding approach |
| 3. Data & Features | 5-7 min | Sessions, sliding window, no listing attributes |
| 4. Model Architecture | 8-10 min | Listing2Vec, loss improvements |
| 5. Evaluation Metrics | 3-5 min | Avg rank of booked listing + session book rate |
| 6. Serving Architecture | 8-10 min | Three pipelines, cold start, ANN |
| 7. Follow-ups | 5-8 min | Position bias, seasonality, personalization |

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# === Interview Timeline ===

phases = [
    'Clarify\nRequirements', 'Frame as\nML Problem', 'Data &\nFeatures',
    'Model\nArchitecture', 'Evaluation\nMetrics', 'Serving\nArchitecture', 'Follow-ups'
]
times = [4, 4, 6, 9, 4, 9, 7]
cumulative = np.cumsum([0] + times[:-1])
colors = ['#FFECB3', '#B3E5FC', '#C8E6C9', '#F8BBD0', '#D1C4E9', '#FFCCBC', '#FFF9C4']

fig, ax = plt.subplots(figsize=(14, 3))

for i, (phase, t, start, color) in enumerate(zip(phases, times, cumulative, colors)):
    ax.barh(0, t, left=start, height=0.6, color=color, edgecolor='black', linewidth=1.5)
    ax.text(start + t/2, 0, f'{phase}\n({t} min)', ha='center', va='center', fontsize=8, fontweight='bold')

ax.set_xlim(0, 45)
ax.set_ylim(-0.5, 0.5)
ax.set_xlabel('Time (minutes)', fontsize=12)
ax.set_yticks([])
ax.set_title('45-Minute Interview Timeline: Similar Listings', fontsize=14, fontweight='bold')

for m in range(0, 46, 5):
    ax.axvline(x=m, color='gray', linestyle=':', alpha=0.5)
    ax.text(m, -0.4, f'{m}', ha='center', fontsize=8, color='gray')

plt.tight_layout()
plt.show()

print('Golden rule: Do NOT spend more than 5 min on requirements.')
print('Model Architecture and Serving get the MOST time because that is where you show depth.')

## Phase 1: Clarify Requirements (3-5 minutes)

### The Dialogue

---

**Interviewer**: "Design a similar listings feature for a vacation rental platform like Airbnb."

**Candidate**: "I would love to work on this. Let me start by clarifying the scope."

**Candidate**: "What is the business objective? Are we trying to increase bookings, or user engagement, or something else?"

**Interviewer**: "Increase the number of bookings."

**Candidate**: "Great -- that directly connects to revenue. What does 'similarity' mean in this context? Same neighborhood, same price range, same type of listing?"

**Interviewer**: "Yes -- same neighborhood, city, price range, and similar characteristics."

**Candidate**: "Should the recommendations be personalized per user? For example, should a luxury traveler see different 'similar' listings than a budget traveler?"

**Interviewer**: "For simplicity, treat logged-in and anonymous users equally. No personalization."

**Candidate**: "How many listings are on the platform?"

**Interviewer**: "About 5 million."

**Candidate**: "What data can I use for training? Can I use listing attributes like price and location?"

**Interviewer**: "Use user-listing interactions only. The model should not use listing attributes directly."

**Candidate**: "How quickly should new listings appear in recommendations?"

**Interviewer**: "Within one day of being posted."

---

### Summary Statement (Memorize This)

> "To summarize: we are designing a similar listings feature for a vacation rental platform with 5 million listings. Input is the listing a user is currently viewing; output is a ranked list of similar listings. No personalization. Training uses only behavioral data -- user-listing interactions, not listing attributes. New listings should appear within 24 hours. The business goal is to increase bookings."

## Phase 2: Frame as ML Problem (3-5 minutes)

### The Dialogue

---

**Candidate**: "I would frame this as a **session-based recommendation** problem using **listing embeddings**.

The key insight is that listings clicked sequentially in a browsing session tend to share characteristics -- location, price, type. I can use this co-occurrence signal to learn similarity, without needing explicit listing attributes.

This is directly inspired by **Word2Vec**: just as words appearing in similar sentences have similar meanings, listings appearing in the same browsing sessions are similar.

```
Word2Vec analogy:
  word     = listing
  sentence = browsing session
  context  = nearby clicks in the session
```

**ML Objective**: Accurately predict which listing the user will click next, given the listing they are currently viewing.

**Input**: A listing ID (the one the user is viewing)
**Output**: A ranked list of similar listings, sorted by P(user clicks them)

I chose session-based over traditional recommendation because for vacation rentals, what you searched 5 minutes ago is far more informative than what you searched 6 months ago."

**Interviewer**: "Why not use collaborative filtering or a deep model?"

**Candidate**: "Collaborative filtering models long-term user preferences, which is not ideal here. Users' vacation preferences change completely between trips -- a ski trip in January is irrelevant to their beach search in July. A deep model like a Transformer would work but requires more data and compute. The Skip-gram approach is simple, proven at scale (Airbnb uses it in production), and the embeddings are directly usable for fast ANN retrieval."

---

### Key Phrase to Memorize

> "I frame this as session-based recommendation with listing embeddings. Listings are words, sessions are sentences, and co-occurrence defines similarity. This is the Listing2Vec approach, inspired by Word2Vec and used at Airbnb in production."

In [ ]:
# === Whiteboard Sketch: System Overview ===

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')

def box(ax, x, y, w, h, text, color, fontsize=9):
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                                    facecolor=color, edgecolor='#333', linewidth=2)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=fontsize, fontweight='bold')

def arrow(ax, x1, y1, x2, y2, color='#333'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=2))

# Data flow
box(ax, 0.5, 8.5, 3, 0.8, 'User Click Sessions', '#E3F2FD', 10)
box(ax, 0.5, 6.5, 3, 1.0, 'Training\nListing2Vec\n(Skip-gram + neg sampling)', '#BBDEFB', 9)
arrow(ax, 2, 8.5, 2, 7.5)

box(ax, 5, 6.5, 3.5, 1.0, 'Indexing\nPre-compute 5M\nListing Embeddings', '#C8E6C9', 9)
arrow(ax, 3.5, 7, 5, 7)

# Prediction pipeline
box(ax, 10, 8.5, 3.5, 0.8, 'User Views Listing X', '#FFF3E0', 10)
box(ax, 10, 6.5, 3.5, 1.0, '1. Embedding Fetcher\n(lookup or cold-start)', '#FFCCBC', 9)
arrow(ax, 11.75, 8.5, 11.75, 7.5)
arrow(ax, 8.5, 7, 10, 7, '#2E7D32')

box(ax, 10, 4.5, 3.5, 1.0, '2. ANN Search\n(FAISS / ScaNN)\nFind top-K similar', '#F3E5F5', 9)
arrow(ax, 11.75, 6.5, 11.75, 5.5)

box(ax, 10, 2.5, 3.5, 1.0, '3. Re-Ranking\nPrice filter, diversity\nAvailability, city match', '#E0F2F1', 9)
arrow(ax, 11.75, 4.5, 11.75, 3.5)

box(ax, 10, 0.8, 3.5, 0.8, 'Similar Listings Displayed', '#FFF3E0', 10)
arrow(ax, 11.75, 2.5, 11.75, 1.6)

# Loss function annotation
box(ax, 0.5, 4, 8, 1.5,
    'Loss Function (4 Components)\n'
    '1. Positive pairs: push center toward context listings\n'
    '2. Random negatives: push center away from random listings\n'
    '3. Global context: push center toward BOOKED listing\n'
    '4. Hard negatives: push center away from SAME-REGION listings',
    '#FFF9C4', 8)
arrow(ax, 2, 6.5, 4.5, 5.5)

# Metrics
box(ax, 0.5, 1.5, 4, 1.5,
    'Evaluation\n'
    'Offline: Avg rank of booked listing\n'
    'Online: CTR + Session Book Rate',
    '#FCE4EC', 9)

ax.set_title('Whiteboard Sketch: Similar Listings System', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print('Draw this in the first 2 minutes, then walk through each box.')

## Phase 3: Data and Features (5-7 minutes)

### The Dialogue

---

**Candidate**: "For training data, we extract **search sessions** from user interaction logs. A search session is a sequence of clicked listing IDs, followed by the eventually booked listing.

For example: `[L1, L5, L4, L9, L26_booked]` -- the user clicked through L1, L5, L4, L9 and eventually booked L26.

Important design decision: the model uses ONLY this co-occurrence data. It does NOT use listing attributes like price, location, or number of beds. The embeddings implicitly learn these relationships from behavioral patterns."

**Interviewer**: "Why not use listing attributes? Would they not help?"

**Candidate**: "Great question. There are two reasons:

1. **Simplicity**: Co-occurrence data alone has been shown to produce high-quality embeddings (this is the Airbnb KDD 2018 approach). Adding attributes increases model complexity.

2. **Implicit learning**: The model IMPLICITLY learns location and price from behavior. If Miami beach houses always co-occur in sessions, their embeddings will be close together -- even though we never told the model about location.

That said, listing attributes CAN help with cold start. For new listings with no interaction data, we can use attribute similarity to initialize embeddings."

---

**Candidate**: "For constructing training pairs, we use a **sliding window** approach:
- Slide a window of size m across each session
- Center listing + context listings within the window = positive pairs
- Center listing + randomly sampled listings outside = negative pairs
- This is identical to Word2Vec's negative sampling strategy."

---

### Key Phrase to Memorize

> "We extract search sessions from click logs. A sliding window generates positive pairs from co-occurring listings and negative pairs from random sampling. The model uses ONLY behavioral data -- no listing attributes -- but implicitly learns location, price, and type from co-occurrence patterns."

## Phase 4: Model Architecture (8-10 minutes)

### The Dialogue

---

**Candidate**: "The model is a **shallow neural network** -- essentially Skip-gram Word2Vec adapted for listings.

Architecture:
- Embedding matrix: V x d (V = 5 million listings, d = 32 or 64)
- No hidden layers in the simplest form (just an embedding lookup)
- Training: negative sampling with dot product similarity + sigmoid + cross-entropy

The basic loss function has **two shortcomings** that I would fix:

**Problem 1: No booking signal.**
The basic model pushes center listings toward clicked neighbors, but NOT toward the eventually booked listing. Clicking is not the same as booking. 

**Solution: Global Context.** Treat the booked listing as a global positive that stays in every pair throughout the session. This drives embeddings toward booking prediction, not just click prediction.

**Problem 2: Easy negatives.**
Random negatives are mostly from different cities. The model easily distinguishes 'Miami vs Aspen' but fails at 'Miami beach house A vs Miami beach house B.'

**Solution: Hard Negatives.** Sample negative listings from the SAME neighborhood as the center listing. This forces fine-grained discrimination.

The updated loss has four components:"

```
L = -SUM(D_pos)  log(sig(E_c . E_p))       # Positive pairs
    -SUM(D_neg)  log(sig(-E_c . E_n))       # Random negatives
    -SUM(D_book) log(sig(E_c . E_book))     # Global context (BOOKING)
    -SUM(D_hard) log(sig(-E_c . E_hard))    # Hard negatives (SAME REGION)
```

**Interviewer**: "How do these improvements affect the embeddings?"

**Candidate**: "The global context improvement aligns embeddings with the business objective -- bookings, not just clicks. In experiments, it improved the average rank of booked listings by ~20%. Hard negatives improved intra-market discrimination, which is critical because users typically search within one city. Together, they made the embeddings significantly better at predicting what users actually book."

---

### Key Phrase to Memorize

> "The basic loss has two shortcomings. First, it does not incorporate booking signal -- we fix this with global context, treating the booked listing as a persistent positive. Second, random negatives are too easy -- we fix this with hard negatives from the same region. These two improvements are the key differentiators of this system."

In [ ]:
# === Visualize the Two Loss Improvements ===

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Improvement 1: Global Context ---
ax = axes[0]
session = ['L1', 'L5', 'L4', 'L9']
booked = 'L26'

for i, l in enumerate(session):
    color = '#42A5F5'
    rect = plt.Rectangle((i * 1.8, 2), 1.3, 0.6, facecolor=color, edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(i * 1.8 + 0.65, 2.3, l, ha='center', va='center', fontsize=10, fontweight='bold', color='white')

# Booked listing (global)
booked_rect = plt.Rectangle((2.5, 0.3), 1.5, 0.6, facecolor='#FFD54F', edgecolor='black', linewidth=2)
ax.add_patch(booked_rect)
ax.text(3.25, 0.6, f'{booked}\n(BOOKED)', ha='center', va='center', fontsize=9, fontweight='bold')

for i in range(len(session)):
    ax.annotate('', xy=(3.25, 0.95), xytext=(i * 1.8 + 0.65, 1.95),
                arrowprops=dict(arrowstyle='->', color='#E53935', lw=1.5, linestyle='--'))

ax.set_xlim(-0.5, 8)
ax.set_ylim(-0.2, 3)
ax.set_title('Improvement 1: Global Context\n(Booked listing = persistent positive)', fontsize=12, fontweight='bold')
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

# --- Improvement 2: Hard Negatives ---
ax = axes[1]

# Two regions
miami = plt.Circle((2.5, 2.5), 1.8, fill=True, facecolor='#FFCDD2', edgecolor='#E53935', linewidth=2, linestyle='--')
ax.add_patch(miami)
ax.text(2.5, 4.0, 'Miami', fontsize=12, fontweight='bold', ha='center', color='#C62828')

nyc = plt.Circle((6.5, 2.5), 1.8, fill=True, facecolor='#BBDEFB', edgecolor='#1565C0', linewidth=2, linestyle='--')
ax.add_patch(nyc)
ax.text(6.5, 4.0, 'NYC', fontsize=12, fontweight='bold', ha='center', color='#0D47A1')

# Center listing
ax.scatter([2.5], [2.5], c='#E53935', s=200, zorder=5, edgecolors='black', linewidth=2)
ax.text(2.5, 1.8, 'Center', ha='center', fontsize=9, fontweight='bold')

# Easy negative (NYC)
ax.scatter([6.5], [2.5], c='#1565C0', s=150, zorder=5, marker='x', linewidth=3)
ax.text(6.5, 1.8, 'Easy neg\n(different city)', ha='center', fontsize=8, color='#1565C0')

# Hard negative (same city)
ax.scatter([3.3], [3.3], c='#C62828', s=150, zorder=5, marker='x', linewidth=3)
ax.text(3.3, 3.7, 'HARD neg\n(same city!)', ha='center', fontsize=8, color='#C62828', fontweight='bold')

ax.set_xlim(0, 9)
ax.set_ylim(0, 5)
ax.set_title('Improvement 2: Hard Negatives\n(Same region, NOT in context)', fontsize=12, fontweight='bold')
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

print('These two improvements are THE differentiators of this system.')
print('If you explain them clearly in the interview, you are in staff-level territory.')

## Phase 5: Evaluation Metrics (3-5 minutes)

### The Dialogue

---

**Candidate**: "For evaluation, I use both offline and online metrics.

**Offline: Average Rank of Eventually Booked Listing.**
- For each session in the validation set, take the first clicked listing
- Compute cosine similarity to all other listings
- Rank by similarity
- Record where the eventually booked listing appears
- Average across all sessions
- Lower is better (rank 1 = model always ranks the booked listing first)

Why this metric? It directly measures whether the embeddings can surface the listing the user will book. If the new model ranks booked listings higher than the old model, the new embeddings are better.

**Online: Session Book Rate** (primary) + CTR (secondary).
- Session Book Rate = % of sessions that end in a booking
- This is directly tied to the business objective (increase bookings)
- CTR alone is insufficient -- clicks are not bookings"

**Interviewer**: "Why not use MRR or NDCG?"

**Candidate**: "MRR and NDCG are excellent for search ranking. For this particular problem, the average rank of booked listing is more natural because our business cares specifically about bookings, not generic relevance. The booked listing is the ground truth positive, and we want to measure how high it ranks. That said, if we had graded relevance labels from human annotators, NDCG would be a strong complement."

---

### Key Phrase to Memorize

> "My offline metric is the average rank of the eventually booked listing. It directly measures embedding quality for the booking prediction task. Online, session book rate is the primary metric because it ties directly to revenue."

In [ ]:
# === Demonstrate the Offline Metric ===

# Simulate evaluation
np.random.seed(42)

n_sessions = 8
n_listings = 5000

# Simulate: for each session, where does the model rank the booked listing?
basic_model_ranks = np.random.randint(50, 500, n_sessions)  # Basic model: rank 50-500
improved_model_ranks = np.random.randint(1, 50, n_sessions)  # Improved: rank 1-50

fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(n_sessions)
width = 0.35

bars1 = ax.bar(x - width/2, basic_model_ranks, width, label='Basic Listing2Vec',
               color='#90CAF9', edgecolor='black', linewidth=0.5)
bars2 = ax.bar(x + width/2, improved_model_ranks, width, label='Improved (Global Context + Hard Neg)',
               color='#EF5350', edgecolor='black', linewidth=0.5)

ax.axhline(y=np.mean(basic_model_ranks), color='#1565C0', linestyle='--', linewidth=2,
           label=f'Basic avg rank: {np.mean(basic_model_ranks):.0f}')
ax.axhline(y=np.mean(improved_model_ranks), color='#C62828', linestyle='--', linewidth=2,
           label=f'Improved avg rank: {np.mean(improved_model_ranks):.0f}')

ax.set_xlabel('Session', fontsize=12)
ax.set_ylabel('Rank of Booked Listing (lower = better)', fontsize=12)
ax.set_title('Offline Metric: Average Rank of Eventually Booked Listing', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'S{i+1}' for i in range(n_sessions)])
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

improvement = (1 - np.mean(improved_model_ranks) / np.mean(basic_model_ranks)) * 100
print(f'Basic model: average rank = {np.mean(basic_model_ranks):.0f}')
print(f'Improved model: average rank = {np.mean(improved_model_ranks):.0f}')
print(f'Improvement: {improvement:.0f}% better at ranking the booked listing!')

## Phase 6: Serving Architecture (8-10 minutes)

### The Dialogue

---

**Candidate**: "The serving system has three pipelines.

**1. Training Pipeline (Daily, Offline)**
- Runs overnight using the day's new interaction data
- Warm-starts from the previous model (not training from scratch)
- Constructs new search sessions, generates training pairs with sliding window
- Trains Listing2Vec with the improved loss function
- Output: updated model weights

**2. Indexing Pipeline (After Training, Offline)**
- Computes embeddings for all 5 million listings using the new model
- Builds an ANN index (FAISS, ScaNN, or HNSW) for fast similarity search
- Blue-green deployment: the new index replaces the old one atomically

**3. Prediction Pipeline (Real-Time, Online, ~10ms)**
Three components:

a. **Embedding Fetcher**: Looks up the current listing's embedding from the index. For new listings (cold start), uses the embedding of a geographically nearby listing as a proxy.

b. **Nearest Neighbor Service**: Uses ANN search to find the top-K most similar listings from 5M. ANN trades a tiny amount of accuracy for massive speed (5M vectors in <10ms).

c. **Re-Ranking Service**: Applies user filters (price, dates), removes unavailable listings, enforces diversity (max 2 per host), removes listings in different cities."

**Interviewer**: "How do you handle the cold-start problem?"

**Candidate**: "Three strategies:
1. **Immediate**: Use the average embedding of listings in the same neighborhood. This gives a reasonable proxy -- if other Miami Beach condos cluster together, the average is a decent starting point.
2. **Better**: If we know the listing's price and type, find the most similar KNOWN listing and use its embedding.
3. **Permanent fix**: After 24 hours, the listing has enough interaction data to be included in the daily retraining cycle, and it gets its own learned embedding."

---

### Key Phrase to Memorize

> "Three pipelines: training runs daily overnight, indexing pre-computes all 5M embeddings, prediction serves in ~10ms with embedding fetch, ANN search, and re-ranking. Cold start uses geographic proxy until daily retraining gives the new listing its own embedding within 24 hours."

## Phase 7: Follow-Up Questions (5-8 minutes)

### Common Follow-Ups and Staff-Level Answers

---

**Q1: "How would you handle position bias?"**

> "Listings shown at the top get more clicks regardless of true relevance. Two solutions:
> 1. **Inverse propensity weighting**: Weight each click sample by 1/P(click|position). A click on position 5 counts more than a click on position 1.
> 2. **Position as a training feature**: Include display position as a feature during training, then set it to a neutral value during inference.
> I would start with IPW because it is simpler to implement."

---

**Q2: "How would you incorporate seasonality?"**

> "Vacation rentals are highly seasonal. Beach houses spike in summer, ski cabins in winter. Three approaches:
> 1. **Time-windowed training**: Only train on recent sessions (last 30 days) instead of all-time data.
> 2. **Seasonal embeddings**: Learn separate embedding sets for summer vs winter.
> 3. **Time feature**: Add month-of-year as an additional feature to the model.
> I would start with approach 1 (time-windowed) because it is simplest and naturally emphasizes current seasonal patterns."

---

**Q3: "How would you personalize for logged-in users?"**

> "For logged-in users, we can incorporate their booking history. Airbnb does this with in-session personalization:
> - Compute the average embedding of the user's past booked listings
> - Add this as an extra positive signal during embedding retrieval
> - If they previously booked luxury villas, boost luxury results
> This adapts the 'similar listings' to be similar to both the current listing AND the user's demonstrated preferences."

---

**Q4: "How would you compare this to a graph-based approach?"**

> "Graph-based approaches like DeepWalk or Node2Vec would model listings as nodes and co-occurrences as edges. Random walks on this graph generate 'sentences' similar to our browsing sessions.
> 
> Advantages of graph approach: captures multi-hop relationships (listing A connects to B connects to C). 
> Advantages of session approach: more direct signal, simpler implementation, captures temporal ordering.
> 
> For this problem, session-based is preferred because the temporal sequence (what was clicked BEFORE and AFTER) is more informative than the graph structure alone."

---

**Q5: "What would you improve if you had more time?"**

> "Three things, in order of expected impact:
> 1. **In-session personalization** for logged-in users (incorporate past bookings)
> 2. **Feature-enriched embeddings** that concatenate behavioral embeddings with listing attribute embeddings (price bucket, type, location) -- especially helps cold-start
> 3. **A/B testing framework** with counterfactual logging to detect and correct for exploration bias"

In [ ]:
# === Interview Scoring Rubric ===

rubric = {
    'Requirements': {
        'Junior': 'Jumps to model design',
        'Mid': 'Asks basic questions (scale, data)',
        'Senior': 'Asks about business goal, constraints',
        'Staff': 'Probes cold-start, personalization tradeoffs, summarizes crisply',
    },
    'Problem Framing': {
        'Junior': 'Says "use collaborative filtering"',
        'Mid': 'Mentions embeddings but shallow reasoning',
        'Senior': 'Explains session-based approach with Word2Vec analogy',
        'Staff': 'Contrasts session-based vs traditional, explains WHY short-term > long-term for rentals',
    },
    'Loss Function': {
        'Junior': 'Uses basic cross-entropy',
        'Mid': 'Implements negative sampling correctly',
        'Senior': 'Identifies one of the two improvements',
        'Staff': 'Explains BOTH improvements (global context + hard negatives) and their business impact',
    },
    'Evaluation': {
        'Junior': 'Says accuracy or F1',
        'Mid': 'Uses recall or precision',
        'Senior': 'Uses MRR or avg rank',
        'Staff': 'Avg rank of booked listing + session book rate, explains WHY each metric was chosen',
    },
    'Serving + Cold Start': {
        'Junior': 'Skips serving entirely',
        'Mid': 'Mentions ANN search',
        'Senior': 'Three-pipeline architecture, mentions cold start',
        'Staff': 'Full pipeline with latency budget, cold-start strategies ranked by quality, daily retraining cycle',
    },
}

print('=== Interview Scoring Rubric: Similar Listings ===')
print('\nWhat separates each level:\n')

for area, levels in rubric.items():
    print(f'--- {area} ---')
    for level, description in levels.items():
        marker = '>>>' if level == 'Staff' else '   '
        print(f'  {marker} {level}: {description}')
    print()

## The 30-Second Pitch (Memorize This)

If asked to summarize your design in 30 seconds:

> "I designed a session-based similar listing recommendation system using **Listing2Vec** -- a Word2Vec-inspired approach where listings are words and browsing sessions are sentences. The model is trained with an **improved loss function** that incorporates the eventually booked listing as a global context and uses hard negatives from the same geographic region for fine-grained discrimination. Embeddings for all 5 million listings are pre-computed daily and stored in an **ANN index** for sub-10ms retrieval. A **re-ranking service** applies business rules like price filters, availability, and host diversity. For evaluation, I use the **average rank of the booked listing** offline and **session book rate** online."

---

## Key Design Decisions to Defend

| Decision | Defense |
|----------|----------|
| Session-based over traditional | Short-term intent > long-term preferences for vacation rentals |
| Skip-gram (not deep model) | Simple, proven at scale (Airbnb production), embeddings usable for ANN |
| No listing attributes in training | Co-occurrence implicitly learns location/price; keeps model simple |
| Global context (booked listing) | Aligns embeddings with business objective (bookings, not just clicks) |
| Hard negatives (same region) | Forces fine-grained intra-market discrimination |
| Daily retraining | Captures new listings, seasonal shifts, behavior changes |
| Avg rank of booked listing | Directly measures embedding quality for the booking prediction task |

## Final Checklist Before the Interview

Go through this checklist to make sure you can speak confidently about each topic:

- [ ] Can I explain the Word2Vec to Listing2Vec analogy clearly?
- [ ] Can I draw the three-pipeline architecture from memory?
- [ ] Can I explain BOTH loss function improvements (global context + hard negatives)?
- [ ] Do I know WHY session-based > traditional for vacation rentals?
- [ ] Can I explain the sliding window and negative sampling process?
- [ ] Do I know three cold-start strategies and when to use each?
- [ ] Can I explain position bias and how to address it?
- [ ] Do I know the offline metric (avg rank of booked) and why it was chosen?
- [ ] Can I explain the daily retraining cycle (collect -> train -> index -> deploy)?
- [ ] Can I give the 30-second pitch without hesitation?

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)